In [ ]:
from pyspark.sql.session import SparkSession

# spark = SparkSession.builder \
#     .appName("MinIO Write") \
#     .master("local[*]") \
#     .config("spark.eventLog.enabled", "false") \
#     .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") \
#     .config("spark.log.level", "ERROR") \
#     .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
#     .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.13:1.4.2,org.scala-lang:scala-library:2.13.8") \
#     .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
#     .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
#     .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
#     .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
#     .config("spark.hadoop.fs.s3a.path.style.access", "true") \
#     .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
#     .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
#     .getOrCreate()
    
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MinIO Write") \
    .master("local[*]") \
        .config(
            "spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.13:1.5.0"
        ) \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.eventLog.enabled", "false") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
        .config("spark.sql.catalogImplementation", "in-memory") \
    .getOrCreate()

# Criar um DataFrame de exemplo
data = [("Alice", 29), ("Bob", 35), ("Catherine", 24), ("novoteste", 30)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)

# Exibir os dados para verificar o conteúdo
df.show()

# Escrever o DataFrame no MinIO em formato Parquet
output_path = "s3a://testespark/notebook/"  # Substitua pelo nome do bucket configurado no MinIO
df.repartition(10).write.mode("overwrite").parquet(output_path)

print(f"Data written successfully to {output_path}")

df = spark.read.parquet(f"{output_path}")

df.show()
